# Cross-Context Mode Conservation

Thin package-backed workflow for comparing dynamic modes across contexts. Reusable mode matching, switch classification, and plotting live in `chronopack.tl` and `chronopack.pl`.

In [ ]:
from pathlib import Path
import os

import anndata as ad
import chronopack as cp

workspace_notebook_dir = Path.cwd().resolve()
experiment_dir = workspace_notebook_dir.parent if workspace_notebook_dir.name == "notebooks" else workspace_notebook_dir
OUTPUT_DIR = Path(os.environ.get(
    "CHRONOPACK_DIMRED_OUTPUT_DIR",
    experiment_dir / "self_contained_outputs/global_analysis_extended",
)).expanduser().resolve()
MODE_OUTDIR = OUTPUT_DIR / "mode_conservation"
MODE_OUTDIR.mkdir(parents=True, exist_ok=True)

adata_path = OUTPUT_DIR / "gene_trajectory_clusters.h5ad"
if not adata_path.exists():
    raise FileNotFoundError(
        f"Missing {adata_path}. Run DimensionalityReduction&Clustering.self_contained.ipynb first "
        "to generate standardized gene-trajectory clustering AnnData."
    )
adata = ad.read_h5ad(adata_path)
print(adata)

## K562 TNF-alpha vs THP-1 LPS

In [ ]:
k562_thp1 = cp.tl.mode_conservation(
    adata,
    adata,
    source_context="K562a",
    target_context="THP1",
    source_label="K562 TNF-alpha",
    target_label="THP-1 LPS",
    top_k=300,
)
k562_thp1.write(MODE_OUTDIR, prefix="K562a_vs_THP1")
cp.pl.mode_conservation(k562_thp1, output=MODE_OUTDIR / "K562a_vs_THP1.conservation.svg")
cp.pl.mode_sankey(k562_thp1, output=MODE_OUTDIR / "K562a_vs_THP1.sankey.svg")
k562_thp1.conservation_scores

## K562 TNF-alpha vs K562 TNF-alpha + Puromycin

In [ ]:
k562_puro = cp.tl.mode_conservation(
    adata,
    adata,
    source_context="K562a",
    target_context="K562_puro",
    source_label="K562 TNF-alpha",
    target_label="K562 TNF-alpha + puromycin",
    top_k=300,
)
k562_puro.write(MODE_OUTDIR, prefix="K562a_vs_K562_puro")
cp.pl.mode_conservation(k562_puro, output=MODE_OUTDIR / "K562a_vs_K562_puro.conservation.svg")
cp.pl.mode_sankey(k562_puro, output=MODE_OUTDIR / "K562a_vs_K562_puro.sankey.svg")
k562_puro.conservation_scores

## Switcher Tables

In [ ]:
k562_thp1.switchers.head()